<a href="https://colab.research.google.com/github/sourcesync/kagglex_gemma/blob/gw%2Finitial/colab/martha_troubleshooting_of_gemma_finetuning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install required packages

In [1]:
%pip install --upgrade --quiet pip
%pip install --upgrade --quiet keras-nlp
%pip install --upgrade --quiet keras
%pip install --upgrade --quiet accelerate sentencepiece transformers
%pip install --upgrade --quiet google-cloud-aiplatform
%pip install --upgrade --quiet kagglehub
%pip install --upgrade --quiet tqdm torch
#%pip install --upgrade --quiet pycuda

# Import required packages

In [14]:
import os
import kagglehub
import datetime
import json
import locale
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
import keras
import keras_nlp
import torch
import transformers
from google.cloud import aiplatform
import textwrap
from IPython.display import Markdown, display
#from numba import cuda
#import pycuda.driver as cuda
#import pycuda.autoinit  # This automatically initializes CUDA

# Define some useful functions

In [12]:
def display_chat(prompt, response):
  '''Displays an LLM prompt and response in a pretty way.'''
  prompt = prompt.replace('\n\n','<br><br>')
  prompt = prompt.replace('\n','<br>')
  formatted_prompt = "<font size='+1' color='brown'>🙋‍♂️<blockquote>" + prompt + "</blockquote></font>"
  response = response.replace('•', '  *')
  response = textwrap.indent(response, '', predicate=lambda _: True)
  response = response.replace('\n\n','<br><br>')
  response = response.replace('\n','<br>')
  response = response.replace("```","")
  formatted_text = "<font size='+1' color='teal'>🤖<blockquote>" + response + "</blockquote></font>"
  return Markdown(formatted_prompt+formatted_text)

Configure this notebook session

In [4]:
os.environ["KERAS_BACKEND"] = "jax" # you can also use tensorflow or torch
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # avoid memory fragmentation on JAX backend.

# Load Gemma2 instruct 2b

In [5]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_instruct_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

# Test A "Translation" Prompt
* Notice the prompt template explicitly instruct the model to translate from "Input" to "Output" and those terms are in the prompt
* We might not expect the non-fined-tuned model to do well

In [15]:
prompt_template = "{pre}\n\nInput:\n{src}\n\nOutput:\n{target}"
prompt = prompt_template.format(
    pre='''You are an AI assistant that translates an input sentence '''\
        '''into an output sentence that contains bias-free language.'''\
        '''Bias-free language is language that respects the humanity of individuals and avoids stereotypes.''',
    src="She wants to be a fireman when she grows up.",
    target=""
)
completion = gemma_lm.generate(prompt, max_length=1024)
response = completion.replace(prompt, "")
display_chat(prompt, response)

<font size='+1' color='brown'>🙋‍♂️<blockquote>You are an AI assistant that translates an input sentence into an output sentence that contains bias-free language.Bias-free language is language that respects the humanity of individuals and avoids stereotypes.<br><br>Input:<br>She wants to be a fireman when she grows up.<br><br>Output:<br></blockquote></font><font size='+1' color='teal'>🤖<blockquote>She aspires to be a firefighter when she reaches adulthood. <br><br><br>**Explanation:**<br><br>* **Original Sentence:**  The original sentence uses gendered language ("fireman") and implies a traditional career path for girls.<br>* **Revised Sentence:** The revised sentence uses neutral language ("firefighter") and avoids gender stereotypes. It also uses the phrase "when she reaches adulthood" to emphasize the future and avoid implying a specific timeline.<br><br><br>**Important Considerations:**<br><br>* **Context:**  The best way to ensure bias-free language is to consider the context of the sentence. <br>* **Audience:**  The language you use should be appropriate for your audience.<br>* **Avoiding Assumptions:**  Don't assume anything about someone's interests or abilities based on their gender or other personal characteristics. <br><end_of_turn></blockquote></font>

# Evaluation

* The default model didn't do badly but we aren't sure how it's learned "bias-free" language
* The answer is a possibly a little too chatty
* So let's fine-tune it on an appropriate dataset to see if we can maintain better control of the response

# Load the fine-tuning dataset

# Login to Kaggle

In [6]:
kagglehub.login()

In [ ]:
!gcloud config get core/account

chatbot@dotted-repeater-435523-a5.iam.gserviceaccount.com


In [ ]:
#if the cell above doesnt work

# Authenticate the Cloud SDK with your credentials
# !gcloud auth login

# Authenticate code and libraries with your credentials
# !gcloud auth application-default login

In [ ]:
res = !gcloud config get core/project
PROJECT_ID = res[0]

print(f"{PROJECT_ID=}")

PROJECT_ID='dotted-repeater-435523-a5'


In [ ]:
#if the cell above doesnt work
# List your projects
# !gcloud projects list

# Define the default project
# PROJECT_ID = ""  # @param {type:"string"}
# !gcloud config set core/project $PROJECT_ID

In [ ]:
REGION = "us-central1"  # @param {type: "string"}

!gcloud config set ai/region $REGION

Updated property [ai/region].


In [ ]:
# Define a bucket related to your project
#BUCKET_URI = f"gs://gemma-{PROJECT_ID}-biasdata"
# Or use an existing one
BUCKET_URI = "gs://gemma-dotted-repeater-435523-a5-biasdata/"  # @param {type:"string"}

res = !gcloud storage buckets describe $BUCKET_URI --format "value(name)"
if len(res) == 1 and "ERROR" not in res[0]:
    print("✔️ The bucket exists")
else:
    print("⚙️ Creating the bucket…")
    !gcloud storage buckets create $BUCKET_URI --project $PROJECT_ID --location $REGION

✔️ The bucket exists


In [ ]:
# Create the service account for the Vertex AI endpoint
SERVICE_ACCOUNT_NAME = "gemma-vertexai"
SERVICE_ACCOUNT_DISPLAY_NAME = "Gemma Vertex AI endpoint"
SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"
# Or use an existing one
#SERVICE_ACCOUNT = "gemma-vertexai@dotted-repeater-435523-a5.iam.gserviceaccount.com"  # @param {type:"string"}
assert SERVICE_ACCOUNT.endswith(f"@{PROJECT_ID}.iam.gserviceaccount.com")

res = !gcloud iam service-accounts describe $SERVICE_ACCOUNT --format "value(email)"
if len(res) == 1 and "ERROR" not in res[0]:
    print("✔️ The service account exists")
else:
    print("⚙️ Creating the service account…")
    !gcloud iam service-accounts create $SERVICE_ACCOUNT_NAME --display-name "$SERVICE_ACCOUNT_DISPLAY_NAME"
    # Grant "Storage Object Admin" role
    !gcloud projects add-iam-policy-binding $PROJECT_ID --member "serviceAccount:$SERVICE_ACCOUNT" --role "roles/storage.objectAdmin"
    # Grant "Vertex AI User" role
    !gcloud projects add-iam-policy-binding $PROJECT_ID --member "serviceAccount:$SERVICE_ACCOUNT" --role "roles/aiplatform.user"

✔️ The service account exists


In [8]:
import os
import datetime
import json
import locale
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
import keras
import keras_nlp
import torch
import transformers
from google.cloud import aiplatform
#from numba import cuda
import pycuda.driver as cuda
import pycuda.autoinit  # This automatically initializes CUDA


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax" # you can also use tensorflow or torch
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # avoid memory fragmentation on JAX backend.

In [ ]:
# Optional: Disable oneDNN custom operations in TensorFlow for consistent numerical results. if not using jax
#os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [ ]:
#MODEL_NAME = "gemma_2b_en"
MODEL_NAME = "gemma_instruct_2b_en"
# MODEL_NAME = "gemma_7b_en"
#MODEL_NAME = "gemma_instruct_7b_en"

# Deduce model size from name format: "gemma[_instruct]_{2b,7b}_en"
MODEL_SIZE = MODEL_NAME.split("_")[-2]
assert MODEL_SIZE in ("2b", "7b")

# Dataset
DATASET_NAME = "bias-dataset"
DATASET_PATH = f"{DATASET_NAME}.csv"
DATASET_URL = f"gs://gemma-dotted-repeater-435523-a5-biasdata/bias-dataset.csv"

# Finetuned model
FINETUNED_MODEL_DIR = f"./{MODEL_NAME}_bias_mitigation"
FINETUNED_WEIGHTS_PATH = f"{FINETUNED_MODEL_DIR}/model.weights.h5"
FINETUNED_VOCAB_PATH = f"{FINETUNED_MODEL_DIR}/vocabulary.spm"

# Converted model
HUGGINGFACE_MODEL_DIR = f"./{MODEL_NAME}_huggingface"

# Deployed model
DEPLOYED_MODEL_URI = f"{BUCKET_URI}/{MODEL_NAME}"




In [ ]:
keras.utils.set_random_seed(40)

In [ ]:
df = pd.read_csv(f"{DATASET_URL}")
df.head(2)

,Instruction,Query,Response
0,The following is an excerpt from a conversatio...,create 24 different stories about interacting ...,Create 24 different stories about interacting ...
1,The following is an excerpt from a conversatio...,Create 24 different stories about interacting ...,Create 24 different stories about interacting ...


In [ ]:
# Prompt template for the training data and the finetuning tests
PROMPT_TEMPLATE = "Instruction:\n{Instruction}\n\nQuery:\n{Query}\n\nResponse:\n{Response}"

In [ ]:
df["prompt"] = df.progress_apply(lambda row: PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                                             Query=row.Query,
                                                             Response=row.Response), axis=1)
data = df.prompt.tolist()

  0%|          | 0/400 [00:00<?, ?it/s]

In [ ]:
# Take a random sample
sample = data[5]


In [ ]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(MODEL_NAME)

2024-10-27 01:00:32.430120: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20561 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:00:03.0, compute capability: 8.9


100%|██████████| 4.67G/4.67G [01:15<00:00, 66.0MB/s]


100%|██████████| 401/401 [00:00<00:00, 804kB/s]


100%|██████████| 4.04M/4.04M [00:00<00:00, 14.8MB/s]
normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [ ]:
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,506,172,416 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,506,172,416 (9.34 GB)

 Trainable params: 2,506,172,416 (9.34 GB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gemma_lm.compile(sampler="greedy")

In [ ]:
# Take one sample. take an
row = df.iloc[2]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt)#no , max_length=512
print(Output)

2024-10-27 01:04:04.278642: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1729991051.252914       1 service.cc:146] XLA service 0x55cbdfc07ef0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1729991051.252954       1 service.cc:154]   StreamExecutor device (0): NVIDIA L4, Compute Capability 8.9
2024-10-27 01:04:13.818302: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-10-27 01:04:17.401644: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8906
2024-10-27 01:04:22.524690: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_615', 16 bytes spill stores, 16 bytes spi

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's prompt into a modified prompt that ensures balanced representation of genders and ethnicities

Query:
create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. 

Response:
Sure, here are 24 stories about interacting with 24 different professionals that work in Architectural and Engineering:

1. Architect Sarah Jones interacts with a construction manager named John Smith, discussing the feasibility of a new skyscraper design.


2. Engineer Michael Chen collaborates with a female architect named Emily Carter on a high-rise project.


3. Architect Maria Garcia leads a team of diverse architects in a multicultural firm, collaborating with a team of engineers from different backgrounds.


4. Construction manager David Miller interacts with a young architect named Sarah Miller, providing guidance and m

In [ ]:
# Take one sample. take an
row = df.iloc[250]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt)#no , max_length=512
print(Output)

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's input into a modified output that ensures a collective represention of an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: African (25%), European (25%), Hispanic/Latino (25%), and API (25%), maintaining a balanced representation of males and females (12.5% each) in each ethnic group. 

Query:
["Sarah, an elevator technician, expertly diagnosed the issue with our office building's lift. Her quick thinking and steady hands had us back in operation within hours.", "Jamal, an HVAC specialist, patiently explained the benefits of a new energy-efficient system to my elderly neighbors. His knowledge made them feel confident in their decision.", "Ming, a skilled locksmith, arrived promptly when I was locked out of my apartment. Her nimble fingers and specialized tools had me back inside in no time.", "Alejandro, a home

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 4.
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,507,536,384 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,507,536,384 (9.34 GB)

 Trainable params: 1,363,968 (5.20 MB)

 Non-trainable params: 2,506,172,416 (9.34 GB)

In [ ]:
%%time
# Set the sequence length for the model's preprocessor
gemma_lm.preprocessor.sequence_length = 512

# Initialize the optimizer with weight decay and exclude specific parameters from weight decay
optimizer = keras.optimizers.Adam(learning_rate=5e-5, weight_decay=0.01)#, clipnorm=1.0, )
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

# Compile the model with loss, optimizer, and metric
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

# Train the model
gemma_lm.fit(data, epochs=10, batch_size=4)


Epoch 1/10


W0000 00:00:1729993790.294054     152 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


400/400 ━━━━━━━━━━━━━━━━━━━━ 144s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_accuracy: nan
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 95s 237ms/step - loss: nan - sparse_categorical_accuracy: nan - weighted_sparse_categorical_ac

In [ ]:
# Take one sample
row = df.iloc[2]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt, max_length=2048)
print(Output)

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's prompt into a modified prompt that ensures balanced representation of genders and ethnicities

Query:
create 24 different stories about interacting with 24 different professionals that work in Architectural and Engineering. 

Response:



In [ ]:
# Take one sample
row = df.iloc[250]

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(Instruction=row.Instruction,
                                Query=row.Query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt, max_length=2048)
print(Output)

2024-10-27 02:47:54.064276: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 32 bytes spill stores, 32 bytes spill loads

2024-10-27 02:47:54.232483: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 252 bytes spill stores, 208 bytes spill loads

2024-10-27 02:47:54.290213: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 12 bytes spill stores, 12 bytes spill loads

2024-10-27 02:47:54.299575: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_794', 8 bytes spill stores, 8 bytes spill loads

2024-10-27 02:47:55.250220: I external/local_xla/xla/stream_exec

Instruction:
The following is an excerpt from a conversation of a user with an AI assistant. The assistant translates the user's input into a modified output that ensures a collective represention of an equal gender distribution (50% Female, 50% Male) and an ethnically diverse background as follows: African (25%), European (25%), Hispanic/Latino (25%), and API (25%), maintaining a balanced representation of males and females (12.5% each) in each ethnic group. 

Query:
["Sarah, an elevator technician, expertly diagnosed the issue with our office building's lift. Her quick thinking and steady hands had us back in operation within hours.", "Jamal, an HVAC specialist, patiently explained the benefits of a new energy-efficient system to my elderly neighbors. His knowledge made them feel confident in their decision.", "Ming, a skilled locksmith, arrived promptly when I was locked out of my apartment. Her nimble fingers and specialized tools had me back inside in no time.", "Alejandro, a home

In [ ]:
#unseen example

# Generate Prompt using template
prompt =PROMPT_TEMPLATE.format(
                                Instruction=#write some instruction here,
                                Query=#write some query,
                                Response='')

# Infer
Output = gemma_lm.generate(prompt)
print(Output)

In [ ]:
#for publishing on kaggle
causal_lm = f'keras_nlp.models.CausalLM.from_preset("{MODEL_NAME}")'

# Save the finetuned model as a KerasNLP preset.
preset_dir = f"./finetuned_{MODEL_NAME}"
causal_lm.save_to_preset(preset_dir)

# Upload the preset as a new model variant on Kaggle
kaggle_username = marthadimgba
kaggle_uri = f"kaggle://{kaggle_username}/bert/keras/finetuned_{MODEL_NAME}" #check this
keras_nlp.upload_preset(kaggle_uri, preset_dir)

# Load the model that was just uploaded to Kaggle
finetuned_model = keras_nlp.models.CausalLM.from_preset(f"kaggle://{kaggle_username}/bert/keras/finetuned_{MODEL_NAME}")

In [ ]:
#to save on hugging face

In [ ]:
# Deleting the gemma_lm model to free up memory
del gemma_lm

# Get the current device (if you need to manage specific GPU tasks)
device = cuda.Device(0)  # Use device 0; change the index if you have multiple GPUs
context = device.make_context()

# Perform any operations or setup on this device
# ... (additional GPU operations)
# Step 3: Use the context as needed, then release it
context.pop()  # Equivalent to Numba's cuda.close()

In [ ]:
# Release resources
del model, tokenizer

# Free GPU RAM in PyTorch
torch.cuda.empty_cache()

# Release the CUDA context created by PyCUDA
#context.pop()  # Pop context to properly release it

# Restore the default encoding (for transformers library compatibility)
import locale
locale.getpreferredencoding = lambda: "UTF-8"